In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
PREPATH = '../Data/PreProcessed/'
POSPATH = '../Data/PosProcessed/'
CLASSPATH = '../Data/Classification/'
UDERREVIEWPATH = '../Data/UndeRreview/'

---

### **| Fase 1 ) -** Pré-Processamento

---

In [3]:
df_fair = pd.read_excel(os.path.join(PREPATH, 'PreDatasets.xlsx'))
df_fair.head()

,id,nome,url,repo,is_duplicate,key_value,parent_id
0,CRED-001,Statlog (German Credit Data),https://archive.ics.uci.edu/dataset/144/statlo...,UCI Repository,Yes,144,CRED-002
1,CRED-002,South German Credit,https://www.kaggle.com/datasets/sid321axn/sout...,Kaggle,Ofc,sid321axn/south-german-credit-updated,NaN
2,CRED-003,Default of Credit Card Clients,https://archive.ics.uci.edu/dataset/350/defaul...,UCI Repository,Mod,350,CRED-012
3,CRED-004,Australian Credit Approval,https://www.kaggle.com/datasets/bfueojjsjdjsl/...,Kaggle,No,bfueojjsjdjsl/australian-credit-approval-data-set,NaN
4,CRED-005,Japanese Credit Screening,https://www.kaggle.com/datasets/xiangshan1989/...,Kaggle,No,xiangshan1989/japanese-credit-screening-from-t...,NaN


In [4]:
total_inicial = len(df_fair)
print(f"| 1. Registros totais encontrados via API: {total_inicial}")

df_cleaned = df_fair.dropna(subset=['url']).copy()
total_pos_url = len(df_cleaned)
removidos_url = total_inicial - total_pos_url
print(f"| 2. Removidos por ausência de URL/Documentação: {removidos_url}")
print(f"|    -> Restantes: {total_pos_url}")

mask_originalidade = (df_cleaned['is_duplicate'] == 'No') | (df_cleaned['is_duplicate'] == 'Ofc')
df_cleaned = df_cleaned[mask_originalidade].copy()
total_pos_duplicata = len(df_cleaned)
removidos_duplicata = total_pos_url - total_pos_duplicata
print(f"| 3. Removidos por redundância/duplicação: {removidos_duplicata}")
print(f"|    -> Restantes: {total_pos_duplicata}")

df_cleaned['parent_id'] = df_cleaned['parent_id'].fillna('NaN')
df_cleaned = df_cleaned.reset_index(drop=True)

print(f"| RESULTADO FINAL: {len(df_cleaned)} datasets prontos para análise.")

| 1. Registros totais encontrados via API: 81
| 2. Removidos por ausência de URL/Documentação: 2
|    -> Restantes: 79
| 3. Removidos por redundância/duplicação: 18
|    -> Restantes: 61
| RESULTADO FINAL: 61 datasets prontos para análise.


---

##### | 1.1 ) - Kaggle

In [5]:
df_cleaned_kaggle = df_cleaned[df_cleaned['repo'] == 'Kaggle']
df_cleaned_kaggle.head()

,id,nome,url,repo,is_duplicate,key_value,parent_id
0,CRED-002,South German Credit,https://www.kaggle.com/datasets/sid321axn/sout...,Kaggle,Ofc,sid321axn/south-german-credit-updated,NaN
1,CRED-004,Australian Credit Approval,https://www.kaggle.com/datasets/bfueojjsjdjsl/...,Kaggle,No,bfueojjsjdjsl/australian-credit-approval-data-set,NaN
2,CRED-005,Japanese Credit Screening,https://www.kaggle.com/datasets/xiangshan1989/...,Kaggle,No,xiangshan1989/japanese-credit-screening-from-t...,NaN
3,CRED-007,Polish Companies Bankruptcy,https://www.kaggle.com/datasets/stealthtechnol...,Kaggle,No,stealthtechnologies/predict-bankruptcy-in-poland,NaN
9,CRED-022,Home Credit Default Risk,https://www.kaggle.com/c/home-credit-default-risk,Kaggle,No,competitions/home-credit-default-risk/data,NaN


---

##### | 1.2 ) - UCI Repository	

In [6]:
df_cleaned_uci = df_cleaned[df_cleaned['repo'] == 'UCI Repository']
df_cleaned_uci.head()

,id,nome,url,repo,is_duplicate,key_value,parent_id


---

##### | 1.3 ) - Open ML

In [7]:
df_cleaned_openml = df_cleaned[df_cleaned['repo'] == 'OpenML']
df_cleaned_openml.head()

,id,nome,url,repo,is_duplicate,key_value,parent_id
4,CRED-008,Qualitative Bankruptcy,https://www.kaggle.com/datasets/jagadeesh23/qu...,OpenML,No,1495,NaN
5,CRED-010,credit-approval,https://www.openml.org/d/29,OpenML,Ofc,29,NaN
6,CRED-012,default-of-credit-card-clients v1,https://www.openml.org/d/42477,OpenML,Ofc,42477,NaN
7,CRED-020,LoanDefaultPrediction,https://www.openml.org/d/44067,OpenML,No,44067,NaN
8,CRED-021,bank-marketing,https://www.openml.org/d/44074,OpenML,No,44074,NaN


---

### **| Fase 2 ) -** Extração

---

In [8]:
import os
import shutil
import openml
import locale
import zipfile
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo
from kaggle.api.kaggle_api_extended import KaggleApi

---

##### A ) - Funções Auxiliares e de Tratamento

In [9]:
def is_encrypted(mean_length, threshold=5):
    """Verifica ofuscação pelo tamanho médio do nome das colunas."""
    return mean_length < threshold if pd.notnull(mean_length) else 'N/A'


def get_columns_size_mean(columns):
    """Calcula o tamanho médio dos nomes das colunas."""
    if len(columns) == 0: return 0
    return np.mean([len(str(c)) for c in columns])


---

##### B1 ) - Kaggle

In [10]:
def process_kaggle_datasets(df_input, path='../Data/Buffer'):
    api = KaggleApi()
    api.authenticate()
    results_list = []
    erros_list = []

    for index, row in df_input.iterrows():
        dataset_id = row['id']
        ref = str(row['key_value'])
        os.makedirs(path, exist_ok=True)
        
        base_row_data = {
            'id': dataset_id, 
            'Dataset_Title': row['nome'], 
            'URL': row['url'],
        }

        try:
            if dataset_id in ['CRED-049', 'CRED-059', 'CRED-057', 'CRED-064', 'CRED-038', 'CRED-030']:
                raise Exception("Dataset massivo/complexo. Processamento bloqueado para proteger a memória RAM.")
            
            if ref.startswith('competitions/'):
                comp_name = ref.replace('competitions/', '')
                api.competition_download_files(comp_name, path=path)
            else:
                try:
                    api.dataset_download_files(ref, path=path, unzip=True)
                except Exception as e:
                    print(f"| Aviso: Falha no download padrão. Tentando modo competição para {ref}...")
                    api.competition_download_files(ref, path=path)

            for item in os.listdir(path):
                if item.endswith('.zip'):
                    caminho_zip = os.path.join(path, item)
                    try:
                        with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
                            zip_ref.extractall(path)
                        os.remove(caminho_zip)
                    except zipfile.BadZipFile:
                        print(f"| Aviso: O arquivo {item} está corrompido ou não é um zip válido.")

            files = [f for root, _, f_names in os.walk(path) for f in f_names if f.endswith('.csv')]
            
            if not files:
                raise Exception("Nenhum arquivo CSV encontrado após o download.")

            total_size_mb = 0.0
            all_unique_columns = set() 
            
            all_numeric_columns = set()
            all_categorical_columns = set()
            
            is_dataset_encrypted = False
            file_names_list = []

            for file_name in files:
                file_path = os.path.join(path, file_name)
                
                tamanho_bytes = os.path.getsize(file_path)
                total_size_mb += (tamanho_bytes / (1024**2))
                
                df_temp = pd.read_csv(file_path, low_memory=False)
                print(f"Processando: {file_name}")
                
                colunas_atuais = df_temp.columns.values
                all_unique_columns.update(colunas_atuais)
                
                colunas_numericas = df_temp.select_dtypes(include=['number']).columns
                colunas_categoricas = df_temp.select_dtypes(exclude=['number']).columns
                
                all_numeric_columns.update(colunas_numericas)
                all_categorical_columns.update(colunas_categoricas)
                
                if is_encrypted(mean_length=get_columns_size_mean(colunas_atuais), threshold=5):
                    is_dataset_encrypted = True
                    
                file_names_list.append(file_name)

            file_data = base_row_data.copy()
            file_data.update({
                'Files_Count': len(files),
                'Files_Names': " | ".join(file_names_list), 
                'Total_Size_MB': round(total_size_mb, 2),
                'is_encrypted': is_dataset_encrypted,
                'Columns': list(all_unique_columns), 
                'Unique_Columns_Count': len(all_unique_columns), 
                'Total_Numeric_Cols': len(all_numeric_columns),
                'Total_Categorical_Cols': len(all_categorical_columns)
            })
            
            results_list.append(file_data)
                
        except Exception as e:
            print(f"| {dataset_id} -> Erro Técnico: {e}")
            base_row_data['Erro'] = str(e)
            erros_list.append(base_row_data)
            
        finally:
            shutil.rmtree(path, ignore_errors=True)

    return pd.DataFrame(results_list), pd.DataFrame(erros_list)

---

##### B2 ) - OpenML

In [11]:
def process_openml_datasets(df_input):
    results_list = []
    erros_list = []

    for index, row in df_input.iterrows():
        dataset_id = row['id']
        ref = str(row['key_value'])
        
        base_row_data = {
            'id': dataset_id, 
            'Dataset_Title': row['nome'], 
            'URL': row['url'],
        }
        
        try:
            dataset = openml.datasets.get_dataset(ref)
            
            df_temp, _, _, _ = dataset.get_data(dataset_format="dataframe")
            
            caminhos_possiveis = [
                getattr(dataset, 'data_file', None),      
                getattr(dataset, 'parquet_file', None),   
                getattr(dataset, 'feather_file', None),   
                getattr(dataset, 'data_pickle_file', None) 
            ]
            
            caminhos_validos = [c for c in caminhos_possiveis if c is not None and os.path.exists(c)]
            if caminhos_validos:
                caminho_arquivo = caminhos_validos[0]
                nome_arquivo = os.path.basename(caminho_arquivo)
                tamanho_bytes = os.path.getsize(caminho_arquivo)
                tamanho_disco_mb = round(tamanho_bytes / (1024**2), 2)
            else:
                nome_arquivo = f"openml_data_{ref}"
                tamanho_disco_mb = round(df_temp.memory_usage(deep=False).sum() / (1024**2), 2)

            print(f"Processando: {nome_arquivo}")
            
            colunas_atuais = df_temp.columns.values
            
            all_unique_columns = set(colunas_atuais)
            colunas_numericas = df_temp.select_dtypes(include=['number']).columns
            colunas_categoricas = df_temp.select_dtypes(exclude=['number']).columns
            
            all_numeric_columns = set(colunas_numericas)
            all_categorical_columns = set(colunas_categoricas)

            is_dataset_encrypted = False
            if is_encrypted(mean_length=get_columns_size_mean(colunas_atuais), threshold=5):
                is_dataset_encrypted = True

            file_data = base_row_data.copy()
            file_data.update({
                'Files_Count': 1, 
                'Files_Names': nome_arquivo,
                'Total_Size_MB': tamanho_disco_mb,
                'is_encrypted': is_dataset_encrypted,
                'Columns': list(all_unique_columns), 
                'Unique_Columns_Count': len(all_unique_columns), 
                'Total_Numeric_Cols': len(all_numeric_columns),
                'Total_Categorical_Cols': len(all_categorical_columns)
            })
            
            results_list.append(file_data)
            
            del df_temp 
            
        except Exception as e:
            print(f"| {dataset_id} -> Erro Técnico: {e}")
            base_row_data['Erro'] = str(e)
            erros_list.append(base_row_data)

    return pd.DataFrame(results_list), pd.DataFrame(erros_list)

---

##### B3 ) - UCI

In [12]:
def process_uci_datasets(df_input):
    results_list = []
    for index, row in df_input.iterrows():
        dataset_id = row['id']
        row_data = {
            'id': dataset_id, 'User_Ref': row['key_value'], 'Dataset_Title': row['nome'], 'URL': row['url'],
            'is_duplicate': row['is_duplicate'], 'parent_id': str(row['parent_id']).replace('[', '').replace(']', ''),
            'Source': 'UCI', 'File_Name': 'API_Object'
        }

        try:
            dataset = fetch_ucirepo(id=int(row['key_value']))
            df_temp = dataset.data.features
            if df_temp is None: raise Exception("Funcionalidades (features) não encontradas no dataset UCI.")
            
            columns = df_temp.columns.values
            found_sensitive = [col for col in columns if any(s in str(col).lower() for s in SENSITIVE_TERMS)]
            
            row_data.update({
                'Status_Processamento': 'Sucesso', 'Observacao': 'OK',
                'is_encrypted': is_encrypted(mean_length=get_columns_size_mean(columns), threshold=5),
                'Columns': list(columns), 'Columns_Count': df_temp.shape[1], 'Rows_Count': df_temp.shape[0],
                'Rows_With_Missing': df_temp.isnull().any(axis=1).sum(), 'Missing_Values': df_temp.isnull().sum().sum(),
                'Numeric_Cols': len(df_temp.select_dtypes(include=['number']).columns),
                'Categorical_Cols': len(df_temp.select_dtypes(exclude=['number']).columns),
                'Sensitive_Columns': found_sensitive,
                'Memory_Usage_MB': round(df_temp.memory_usage(deep=True).sum() / (1024**2), 2)
            })
            print(f"| {dataset_id} processado com sucesso.")
            del df_temp # Libera a RAM
            
        except Exception as e:
            print(f"| {dataset_id} -> Erro Técnico: {e}")
            row_data.update({
                'Status_Processamento': 'Erro Técnico', 'Observacao': str(e),
                'is_encrypted': 'N/A', 'Columns': 'N/A', 'Columns_Count': 0, 'Rows_Count': 0,
                'Rows_With_Missing': 0, 'Missing_Values': 0, 'Numeric_Cols': 0, 'Categorical_Cols': 0,
                'Sensitive_Columns': [], 'Memory_Usage_MB': 0
            })
        
        results_list.append(row_data)
    return pd.DataFrame(results_list)

---

### **| Fase 3 ) -** Execução

---

In [13]:
try:
    locale.setlocale(locale.LC_TIME, 'C')
except locale.Error:
    pass

print("| Iniciando FASE 2: Extração Kaggle")
df_info_kaggle, df_erros_kaggle = process_kaggle_datasets(df_input=df_cleaned_kaggle)

print("\n| Iniciando FASE 2: Extração OpenML")
df_info_openml, df_erros_openml = process_openml_datasets(df_input=df_cleaned_openml)

# print("\n--- Iniciando FASE 2: Extração UCI ---")
# df_info_uci = process_uci_datasets(df_input=df_cleaned_uci)

| Iniciando FASE 2: Extração Kaggle
Dataset URL: https://www.kaggle.com/datasets/sid321axn/south-german-credit-updated
Processando: german_credit.csv
Dataset URL: https://www.kaggle.com/datasets/bfueojjsjdjsl/australian-credit-approval-data-set
Processando: Credit_Card_Applications.csv
Dataset URL: https://www.kaggle.com/datasets/xiangshan1989/japanese-credit-screening-from-the-uc-irvine
Processando: credit_approval_uci.csv
Dataset URL: https://www.kaggle.com/datasets/stealthtechnologies/predict-bankruptcy-in-poland
Processando: data.csv
| CRED-022 -> Erro Técnico: 403 Client Error: Forbidden for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles
Dataset URL: https://www.kaggle.com/datasets/wordsforthewise/lending-club
| CRED-023 -> Erro Técnico: [Errno 2] No such file or directory: '../Data/Buffer/accepted_2007_to_2018Q4.csv'
| CRED-025 -> Erro Técnico: 403 Client Error: Forbidden for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/Do

In [14]:
df_final = pd.concat([df_info_kaggle, df_info_openml], ignore_index=True)
df_erros = pd.concat([df_erros_kaggle, df_erros_openml], ignore_index=True)

In [ ]:
df_final = df_final.sort_values(by='id').reset_index(drop=True)
df_erros = df_erros.sort_values(by='id').reset_index(drop=True)
df_final.to_excel(os.path.join(UDERREVIEWPATH, 'PosDatasetUR.xlsx'), index=False)
df_erros.to_csv(os.path.join(UDERREVIEWPATH, 'PosDatasetsErros.csv'), index=False)
print(f"| Total de Registros: {len(df_final)}")
print(f"| Sucessos: {len(df_final)}")
print(f"| Erros/Bloqueios: {len(df_erros)}")

| Total de Registros: 51
| Sucessos: 51
| Erros/Bloqueios: 10
